可以，有官方手册。你要找的是 **Jetson Orin Nano Developer Kit Carrier Board Specification**，里面第 **3.3 节 40-Pin Expansion Header，J12** 就是 40Pin 排针定义。官方说明该排针是 **2×20、2.54 mm 间距**，包含 I2C、SPI、UART、I2S、AU clock 和 GPIO 等接口；所有 40Pin 信号电平都是 **3.3V**，不能把 5V 信号直接接到 GPIO。([NVIDIA Developer][1])

## 1. Jetson Orin Nano 40Pin J12 引脚定义

下表按官方 J12 表 3-3 和图 3-1 整理。注意：这是 **Jetson Orin Nano 官方开发套件载板** 的 40Pin，不一定适用于第三方载板。([NVIDIA Developer][1])

| 物理脚位 | 默认/主要功能             | 可作为 GPIO 时的端口名/说明 |
| ---: | ------------------- | ----------------- |
|    1 | 3.3V 电源             | 供电脚，最大 1A         |
|    2 | 5V 电源               | 供电脚，最大 1A         |
|    3 | I2C1_SDA            | PDD.02            |
|    4 | 5V 电源               | 供电脚，最大 1A         |
|    5 | I2C1_SCL            | PDD.01            |
|    6 | GND                 | 地                 |
|    7 | GPIO09 / Audio MCLK | PAC.06            |
|    8 | UART1_TXD           | PR.02             |
|    9 | GND                 | 地                 |
|   10 | UART1_RXD           | PR.03             |
|   11 | UART1_RTS* / GPIO   | PR.04             |
|   12 | I2S0_SCLK / GPIO    | PH.07             |
|   13 | SPI1_SCK / GPIO     | PY.00             |
|   14 | GND                 | 地                 |
|   15 | GPIO12 / PWM1       | PN.01             |
|   16 | SPI1_CS1* / GPIO    | PY.04             |
|   17 | 3.3V 电源             | 供电脚，最大 1A         |
|   18 | SPI1_CS0* / GPIO    | PZ.06             |
|   19 | SPI0_MOSI / GPIO    | PZ.05             |
|   20 | GND                 | 地                 |
|   21 | SPI0_MISO / GPIO    | PZ.04             |
|   22 | SPI1_MISO / GPIO    | PY.01             |
|   23 | SPI0_SCK / GPIO     | PZ.03             |
|   24 | SPI0_CS0* / GPIO    | PZ.06             |
|   25 | GND                 | 地                 |
|   26 | SPI0_CS1* / GPIO    | PZ.07             |
|   27 | I2C0_SDA            | PDD.00            |
|   28 | I2C0_SCL            | PCC.07            |
|   29 | GPIO01 / GPCLK0     | PQ.05             |
|   30 | GND                 | 地                 |
|   31 | GPIO11 / GPCLK1     | PQ.06             |
|   32 | GPIO07 / PWM7       | PG.06             |
|   33 | GPIO13 / PWM        | PH.00             |
|   34 | GND                 | 地                 |
|   35 | I2S0_FS / GPIO      | PI.02             |
|   36 | UART1_CTS* / GPIO   | PR.05             |
|   37 | SPI1_MOSI / GPIO    | PY.02             |
|   38 | I2S0_DIN / GPIO     | PI.01             |
|   39 | GND                 | 地                 |
|   40 | I2S0_DOUT / GPIO    | PI.00             |

## 2. 你项目里最常用的几个脚

如果 Jetson 和 STM32F407 通信，优先用这组：

| 用途                   | Jetson Orin Nano 40Pin    | STM32F407         |
| -------------------- | ------------------------- | ----------------- |
| Jetson TX → STM32 RX | Pin 8，UART1_TXD           | STM32 某个 USART_RX |
| Jetson RX ← STM32 TX | Pin 10，UART1_RXD          | STM32 某个 USART_TX |
| 共地                   | Pin 6/9/14/20/25/30/34/39 | GND               |

你的 STM32F407 是 3.3V 逻辑，和 Jetson 40Pin 的 3.3V UART 电平兼容；但是 **一定要共地**。不要把 Jetson GPIO 直接接 5V 传感器输出。官方明确说明 40Pin 信号都是 3.3V 电平。([NVIDIA Developer][1])

## 3. 相关官方手册应该看哪几份

最重要的是这三类：

| 文档                                                                        | 用途                                                                                                   |
| ------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------- |
| **Jetson Orin Nano Developer Kit Carrier Board Specification**            | 查 J12 40Pin、相机接口、风扇接口、CAN 预留接口、电源接口等硬件定义。40Pin 表在第 3.3 节。([NVIDIA Developer][1])                     |
| **Jetson Orin Nano Developer Kit User Guide**                             | 开发套件整体使用说明，偏入门和硬件/软件使用入口。官方说明该文档用于从硬件和软件角度使用开发套件。([NVIDIA Developer][2])                             |
| **Jetson Linux Developer Guide：Configuring the Jetson Expansion Headers** | 配置 40Pin 的 pinmux、GPIO、I2C、SPI、I2S 等功能。官方说明很多引脚既可以作为 GPIO，也可以作为 SFIO，例如 I2C/I2S 等。([NVIDIA Docs][3]) |

## 4. 配置和查看 GPIO 的常用命令

Jetson 上不是所有引脚都“插上就能按普通 GPIO 用”。很多脚需要用 **Jetson-IO** 配置 pinmux。官方提供的启动命令是：([NVIDIA Docs][3])

```bash
sudo /opt/nvidia/jetson-io/jetson-io.py
```

查看当前 40Pin 配置：

```bash
sudo /opt/nvidia/jetson-io/config-by-pin.py
sudo /opt/nvidia/jetson-io/config-by-pin.py -p 5
```

官方文档还给了 `gpiod` 的测试方式，例如安装和查看 GPIO 信息：([NVIDIA Docs][3])

```bash
sudo apt install gpiod
gpioinfo
gpiofind PDD.02
gpioget `gpiofind "PDD.02"`
gpioset `gpiofind "PDD.02"`=1
```

## 5. 你需要特别注意的坑

第一，**40Pin 的 5V 脚只能当电源脚，不代表 GPIO 能承受 5V**。GPIO/SPI/UART/I2C 信号全部按 3.3V 处理。([NVIDIA Developer][1])

第二，I2C 引脚和普通 GPIO 引脚电气特性不同。官方表里 I2C 是开漏，带上拉，最大驱动能力按 ±2 mA；很多其他信号经过 TXB0108 电平转换器，官方备注其输出驱动较弱，外接电路不要用强上拉/强下拉。

第三，做小车项目时，Jetson 40Pin 适合接 **STM32 串口、I2C 低速传感器、SPI 外设、简单 GPIO**。不建议把电机驱动、舵机大电流、继电器等负载直接挂到 Jetson GPIO 上，应该经 STM32、驱动板、光耦/三极管/MOS 管隔离控制。

你的当前项目里，Jetson 和 STM32 主通信建议用 **UART1_TXD/RXD，也就是 Pin 8/10**，STM32 负责电机、舵机、OPS、IMU 等实时控制，Jetson 负责视觉和任务决策。

[1]: https://developer.nvidia.com/downloads/assets/embedded/secure/jetson/orin_nano/docs/jetson_orin_nano_devkit_carrier_board_specification_sp.pdf "Jetson Orin Nano Developer Kit Carrier Board Specification"
[2]: https://developer.nvidia.com/embedded/learn/jetson-orin-nano-devkit-user-guide/index.html "Jetson Orin Nano Developer Kit User Guide | NVIDIA Developer"
[3]: https://docs.nvidia.com/jetson/archives/r36.4.3/DeveloperGuide/HR/ConfiguringTheJetsonExpansionHeaders.html "Configuring the Jetson Expansion Headers — NVIDIA Jetson Linux Developer Guide"


In [1]:
!ls /dev/ttyTHS*
!ls /dev/ttyUSB*

/dev/ttyTHS1  /dev/ttyTHS2
ls: 无法访问 '/dev/ttyUSB*': 没有那个文件或目录


In [4]:
import serial
import time

PORT = "/dev/ttyTHS1" # 这个就是USART1的路径

ser = serial.Serial(PORT, 115200, timeout=1)
time.sleep(0.5)

ser.write(b"hello\r\n")
data = ser.readline()

print("recv:", data)

ser.close()

recv: b''


jetson: tx:pin8 rx:pin10
stm32:  tx:PC6  rx:PC7

In [23]:
import serial
import time

PORT = "/dev/ttyTHS1"

ser = serial.Serial(
    port=PORT,
    baudrate=115200,
    timeout=1,
    bytesize=serial.EIGHTBITS,
    parity=serial.PARITY_NONE,
    stopbits=serial.STOPBITS_ONE,
)

time.sleep(0.5)

ser.write(b"hello\r\n")
ser.flush()

ser.close()